# 📄 Notebook 1: PDF Text Extraction for ORNJ Papers

Extract text from ORNJ classification system PDFs using PyMuPDF with Tesseract OCR fallback.

---

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow spacy scikit-learn owlready2 rdflib networkx matplotlib anthropic tqdm
!apt-get install -q tesseract-ocr > /dev/null 2>&1
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/orn-ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'orn-ontology-builder')

## 1. Upload Your ORNJ PDFs

Upload the 16 classification system papers (Marx 1983, Notani 2003, Watson 2024, etc.)

In [ ]:
import os
from google.colab import files
os.makedirs('pdfs', exist_ok=True)
print('Upload ORNJ classification PDFs:')
uploaded = files.upload()
for fn, content in uploaded.items():
    with open(f'pdfs/{fn}', 'wb') as f:
        f.write(content)
    print(f'  Saved: pdfs/{fn} ({len(content):,} bytes)')

## 2. Extract Text

In [ ]:
from src.pdf_extractor import PDFExtractor

extractor = PDFExtractor(ocr_enabled=True)
documents = extractor.extract_from_directory('pdfs/')

print(f'\nExtracted {len(documents)} documents:')
total_chars = 0
for doc in documents:
    chars = len(doc.full_text)
    total_chars += chars
    ocr_pages = sum(1 for p in doc.pages if p.method == 'ocr')
    print(f'  {doc.filename}: {doc.num_pages} pages, {chars:,} chars, {ocr_pages} OCR pages')
print(f'\nTotal: {total_chars:,} characters')

## 3. Save Extracted Text

In [ ]:
import json
os.makedirs('output', exist_ok=True)
extracted = [{'filename': d.filename, 'num_pages': d.num_pages, 'full_text': d.full_text,
    'pages': [{'page': p.page_number, 'text': p.text, 'method': p.method} for p in d.pages]}
    for d in documents]
with open('output/extracted_texts.json', 'w') as f:
    json.dump(extracted, f, indent=2)
print(f'Saved {len(extracted)} documents to output/extracted_texts.json')

---
**Next:** [Notebook 2 — Concept Extraction](02_concept_extraction.ipynb)